In [12]:

import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import datetime as dt
from sqlalchemy import create_engine, text
from sqlalchemy.exc import IntegrityError
from lat_lon_parser import parse
from pytz import timezone
from time import sleep
from dotenv import load_dotenv
import os

# ----------------------------
# LOAD ENV VARIABLES
# ----------------------------
load_dotenv()

True

In [20]:
# ----------------------------
# DATABASE CONNECTION
# ----------------------------
from sqlalchemy import create_engine, text
from sqlalchemy.exc import IntegrityError

schema = "cities_db"
host = "127.0.0.1"
user = "root"
password = os.getenv("MySQL_Password")
port = 3306

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
engine = create_engine(connection_string)


Cities and Population 

In [14]:
# ----------------------------
# 1. POPULATION DATAFRAME
# ----------------------------
def populations_dataframe(cities):
    population_data = []

    for city in cities:
        url = f"https://en.wikipedia.org/wiki/{city}"
        headers = {"User-Agent": "Mozilla/5.0"}

        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")

        # Population
        city_population_clean = None
        try:
            city_population = soup.find(string="Population").find_next("td").get_text()
            city_population_clean = int(city_population.replace(",", ""))
        except:
            pass

        # Area
        city_area_clean = None
        try:
            city_area = soup.find(string="Area").find_next("td").get_text()
            city_area_clean = float(city_area.split("km")[0].strip().replace(",", ""))
        except:
            pass

        today = pd.to_datetime(datetime.today().strftime("%Y-%m-%d"))

        population_data.append({
            "city_name": city,
            "population": city_population_clean,
            "area": city_area_clean,
            "timestamp_population": today
        })

    return pd.DataFrame(population_data)


# ----------------------------
# 2. CITIES DATAFRAME
# ----------------------------
def cities_dataframe(cities):
    city_data = []

    for city in cities:
        url = f"https://en.wikipedia.org/wiki/{city}"
        headers = {"User-Agent": "Mozilla/5.0"}

        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")

        # Latitude
        city_latitude = None
        try:
            city_latitude = soup.find(class_="latitude").get_text()
        except:
            pass

        # Longitude
        city_longitude = None
        try:
            city_longitude = soup.find(class_="longitude").get_text()
        except:
            pass

        # Country
        country = None
        try:
            country_row = soup.find("th", string=lambda x: x and "country" in x.lower())
            if country_row:
                country = country_row.find_next("td").get_text(strip=True)
        except:
            pass

        city_data.append({
            "city_name": city,
            "country": country,
            "latitude": parse(city_latitude) if city_latitude else None,
            "longitude": parse(city_longitude) if city_longitude else None
        })

    return pd.DataFrame(city_data)


# ----------------------------
# 3. UPDATE SQL DATABASE
# ----------------------------
def update_sql_database(cities_df, population_df):

    with engine.begin() as connection:

        # -------- CITIES TABLE --------
        for _, row in cities_df.iterrows():
            try:
                connection.execute(
                    text("""
                        INSERT INTO cities (city_name, country, latitude, longitude)
                        VALUES (:city_name, :country, :latitude, :longitude)
                        ON DUPLICATE KEY UPDATE
                            country = VALUES(country),
                            latitude = VALUES(latitude),
                            longitude = VALUES(longitude)
                    """),
                    row.to_dict()
                )
            except IntegrityError:
                print(f"City '{row['city_name']}' error.")

        # -------- POPULATION TABLE --------
        for _, row in population_df.iterrows():

            try:
                result = connection.execute(
                    text("""
                        SELECT city_id
                        FROM cities
                        WHERE city_name = :city_name
                    """),
                    {"city_name": row["city_name"]}
                ).fetchone()

                if result:
                    city_id = result[0]

                    connection.execute(
                        text("""
                            INSERT INTO population (city_id, population, area, timestamp_population)
                            VALUES (:city_id, :population, :area, :timestamp_population)
                            ON DUPLICATE KEY UPDATE
                                population = VALUES(population),
                                area = VALUES(area)
                        """),
                        {
                            "city_id": city_id,
                            "population": int(row["population"]) if row["population"] else 0,
                            "area": float(row["area"]) if row["area"] else None,
                            "timestamp_population": row["timestamp_population"].date()
                        }
                    )
                else:
                    print(f"City '{row['city_name']}' not found.")

            except IntegrityError:
                print(f"Error for city '{row['city_name']}'.")


# ----------------------------
# 4. MAIN FUNCTION
# ----------------------------
def main(cities):
    cities_df = cities_dataframe(cities)
    population_df = populations_dataframe(cities)
    update_sql_database(cities_df, population_df)


# ----------------------------
# RUN PIPELINE
# ----------------------------

cities = ["Berlin", "Hamburg", "Munich"]
main(cities)

Weather

In [25]:
import pandas as pd
import requests
from pytz import timezone
from datetime import datetime
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

def retrieve_and_send_data():

    # ---------------------------
    # 1. Create DB connection
    # ---------------------------
    def _create_connection():
        schema = "cities_db"
        host = "127.0.0.1"
        user = "root"
        password = os.getenv("MySQL_Password")
        port = 3306

        engine = create_engine(
            f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
        )
        return engine

    # ---------------------------
    # 2. Fetch cities from SQL
    # ---------------------------
    def _fetch_cities(engine):
        return pd.read_sql("SELECT * FROM cities", con=engine)

    # ---------------------------
    # 3. Fetch weather from API
    # ---------------------------
    def _fetch_weather(cities_df):
        berlin_timezone = timezone("Europe/Berlin")
        API_key = os.getenv("WeatherAPI_KEY")
        weather_data = []

        for _, city in cities_df.iterrows():
            city_id = city["city_id"]
            lat = city["latitude"]
            lon = city["longitude"]

            url = (
                f"https://api.openweathermap.org/data/2.5/forecast"
                f"?lat={lat}&lon={lon}&appid={API_key}&units=metric"
            )

            response = requests.get(url)
            weather_json = response.json()

            retrieval_time = datetime.now(berlin_timezone)

            for item in weather_json.get("list", []):

                  weather_main = item.get("main", {})
                  weather_list = item.get("weather")

                  weather_data.append({
                               "city_id": city_id,
                               "forecast_time": item.get("dt_txt"),

                               "outlook": weather_list[0]["description"] if weather_list else None,
 
                                "temperature": weather_main.get("temp"),
                                "feels_like": weather_main.get("feels_like"),

                                "rain_in_last_3h": item.get("rain", {}).get("3h", 0),
                                "wind_speed": item.get("wind", {}).get("speed"),

                                "data_retrieved_at": retrieval_time
                    })
        weather_df = pd.DataFrame(weather_data)
        weather_df["forecast_time"] = pd.to_datetime(weather_df["forecast_time"])
        weather_df["data_retrieved_at"] = pd.to_datetime(weather_df["data_retrieved_at"])
        return weather_df

    # ---------------------------
    # 4. Store into SQL
    # ---------------------------
    def _store_weather(weather_df, engine):
        weather_df.to_sql(
            "weather",
            con=engine,
            if_exists="append",
            index=False
        )

    # ---------------------------
    # 5. Run pipeline
    # ---------------------------
    engine = _create_connection()
    cities_df = _fetch_cities(engine)
    weather_df = _fetch_weather(cities_df)
    _store_weather(weather_df, engine)

    return "Data has been updated successfully"

In [26]:
retrieve_and_send_data()

'Data has been updated successfully'

Airpot

In [21]:
import pandas as pd
from datetime import datetime, timedelta
import requests
from pytz import timezone
from dotenv import load_dotenv
import os
from time import sleep

load_dotenv()

def request_airports_data():
   

    # Pull cities table from database
    cities_df = pd.read_sql('cities', con=connection_string)

    # Create lists to store data
    airports_data = []
    airports_cities_data = []
    # Loop through new cities
    for city in cities:

        row = cities_df.loc[cities_df['city_name'] == city].iloc[0]

        # Construct request
        url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        params = {"withFlightInfoOnly":"true",
                    "lat":row['latitude'],
                    'lon':row['longitude'],
                    'radiusKm':"50",
                    'limit':10}
        headers = {
            "X-RapidAPI-Key": os.getenv("RapidAPI_KEY"),
            "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com",
        }
        # Request from API
        sleep(5)
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        airports_json = response.json()

        # Loop through airports in response
        for airport in airports_json['items']:
            # Gather data for database
            airport_data = {
                'icao': airport['icao'],
                'iata': airport['iata'],
                'airport_name': airport['name'],
                'latitude': airport['location']['lat'],
                'longitude': airport['location']['lon']
            }
            # Add to list of airport data
            airports_data.append(airport_data)

            # Store airport-city connection for bridge table
            airport_city_data = {
                'icao': airport['icao'],
                'city_id': row['city_id'],
            }
            # Add to list of airport-city connections
            airports_cities_data.append(airport_city_data)

    # Convert to data frames
    airports_df = pd.DataFrame(airports_data)
    # Remove any duplicate airports
    airports_df = airports_df.drop_duplicates()
    cities_airports_df = pd.DataFrame(airports_cities_data)

    # Send data to database
    airports_df.to_sql(
        'airports',
        if_exists='append',
        con=connection_string,
        index=False)
    cities_airports_df.to_sql(
        'cities_airports',
        if_exists='append',
        con=connection_string,
        index=False)
   

In [22]:
request_airports_data()

Flight Data

In [23]:
import pandas as pd
from datetime import datetime, timedelta
import requests
from pytz import timezone
from dotenv import load_dotenv
import os
from time import sleep

load_dotenv()

def request_flights_data():

    # Pull airports from database to get corresponding flights
    airports_df = pd.read_sql('airports', con=connection_string)

    flights_data = []

    # Create time windows for 12hr searches
    # Get datetime of tomorrow
    tomorrow = dt.datetime.now(timezone('Europe/Berlin')) + dt.timedelta(days=1)
    # Grab just the date
    tomorrow_date = tomorrow.strftime('%Y-%m-%d')
    # Manually add the rest to get the start and end of the day
    morning_start = f'{tomorrow_date}T00:00'
    morning_end = f'{tomorrow_date}T11:59'
    afternoon_start = f'{tomorrow_date}T12:00'
    afternoon_end = f'{tomorrow_date}T23:59'
    day_parts = [(morning_start, morning_end), (afternoon_start, afternoon_end)]

    # Loop through airports
    for _, row in airports_df.iterrows():
        # Loop for both halves of day
        for time_start, time_end in day_parts:

            # Construct request
            base_url = "https://aerodatabox.p.rapidapi.com/flights/airports"
            path_params = f"/icao/{row['icao']}/{time_start}/{time_end}"
            full_url = base_url + path_params
            params = {
                'withLeg':True,
                'direction':'Arrival',
                'withCancelled':False,
                'withCodeshared':False,
                'withCargo':False,
                'withPrivate':False,
                'withLocation':False
            }
            headers = {
                "X-RapidAPI-Key": os.getenv("RapidAPI_KEY"),
                "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
            }
            # Request data from API
            sleep(5)
            response = requests.get(full_url, headers=headers, params=params)
            if(response.status_code == 200):
                flights_json = response.json()

                # Loop through flights
                for flight in flights_json['arrivals']:

                    # Gather data
                    scheduled_arrival = pd.to_datetime(flight
                                                       ['arrival']
                                                       ['scheduledTime']
                                                       ['local']).replace(tzinfo=None)
                    # If there is no revised time, set it to the scheduled time
                    updated_arrival = pd.to_datetime(
                        flight['arrival']
                        .get('revisedTime', flight['arrival']['scheduledTime'])
                        ['local'],
                    ).replace(tzinfo=None)
                    # Arrange data in dictionary
                    flight_data = {
                        'flight_number': flight['number'],
                        'icao': row['icao'],
                        'scheduled_arrival_time': scheduled_arrival,
                        'updated_arrival_time': updated_arrival,
                        'departure_airport': flight['departure']['airport']['name'],
                    }
                    flights_data.append(flight_data)
            else:
                print(response.status_code, "code for", row['icao'])

    flights_df = pd.DataFrame(flights_data)
    # Send data to database
    try:
        flights_df.to_sql('flights',
                        if_exists='append',
                        con=connection_string,
                        index=False)
    except:
        print("Flights could not be send to SQL database.\n"
              "Maybe you run this function already today.\n"
              "Please come back tomorrow.")

In [24]:
request_flights_data()

429 code for EDDH
429 code for EDDM
429 code for EDDM
429 code for EDDT
429 code for EDDT
